In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from astropy.io import fits


from GalVel_Pop import gal_population, UVW

### Reading data

In [2]:
harps_full = pd.read_csv('harps_params.tsv', sep = '\t')
harps_full.head(5)

,Star,Teff,e_Teff,logg,e_logg,fe_h,e_fe_h,vtur,e_vtur
0,HIP38324,4393,98.0,4.01,0.39,-0.32,0.03,0.44,0.29
1,HD85512,4400,45.0,4.36,0.10,-0.26,0.14,0.02,0.05
2,HD192961,4418,51.0,4.23,0.18,-0.30,0.22,0.02,2.20
3,HD218511,4424,90.0,4.52,0.26,-0.05,0.02,0.26,0.45
4,HIP25612,4433,95.1,4.33,0.35,-0.49,0.06,0.30,0.55


In [3]:
#harps_full.Star.to_csv('harps_names.txt', index = False)

In [4]:
harps_gaia = pd.read_csv('harps_gaia.csv')
harps_gaia.head(5)

,designation,source_id,ra,dec,parallax,parallax_error,pmra,pmra_error,pmdec,pmdec_error,...,distance_gspphot,target_id,target_ra,target_dec,target_parallax,target_pm_ra,target_pm_dec,target_radial_velocity,epoch,target_separation (deg)
0,Gaia DR3 5517008984911824896,5517008984911824896,117.762958,-50.304274,30.807275,0.010410,-140.792583,0.012655,163.940491,0.013387,...,32.4218,HIP38324,117.763938,-50.305003,30.8073,-140.793,163.940,-5.836,2000.0,1.102009e-08
1,Gaia DR3 5412947081295024896,5412947081295024896,147.782211,-43.504882,88.673663,0.017250,461.602652,0.015425,-471.879674,0.019162,...,NaN,HD85512,147.779383,-43.502784,88.6737,461.603,-471.880,-9.560,2000.0,1.085924e-07
2,Gaia DR3 6669657123437047424,6669657123437047424,304.901480,-46.428569,54.044084,0.028673,-375.460767,0.022957,-102.174358,0.020464,...,18.4031,HD192961,304.903901,-46.428115,54.0441,-375.461,-102.174,32.813,2000.0,7.294731e-08
3,Gaia DR3 6391176735363140480,6391176735363140480,347.416897,-67.733773,67.395699,0.019785,-318.045098,0.017669,-214.067791,0.020214,...,NaN,HD218511,347.420627,-67.732822,67.3957,-318.045,-214.068,4.560,2000.0,1.019950e-07
4,Gaia DR3 4621352894595893632,4621352894595893632,82.075400,-81.777340,32.937181,0.010260,-29.644311,0.012080,-441.139859,0.012303,...,30.3643,HIP25612,82.076321,-81.775380,32.9372,-29.644,-441.140,50.037,2000.0,6.035343e-08


In [5]:
harps_gaia_unique = harps_gaia.groupby('target_id', as_index=False).first()
harps_gaia_unique

,target_id,designation,source_id,ra,dec,parallax,parallax_error,pmra,pmra_error,pmdec,...,b,distance_gspphot,target_ra,target_dec,target_parallax,target_pm_ra,target_pm_dec,target_radial_velocity,epoch,target_separation (deg)
0,BD+062932,Gaia DR3 1159108770069883136,1159108770069883136,220.820852,5.827715,12.383062,0.017147,-899.629031,0.021867,0.761438,...,55.927676,80.5051,220.824871,5.827711,12.3831,-899.629,0.761,-144.885,2000.0,1.211333e-07
1,BD+063077,Gaia DR3 4430442044964842368,4430442044964842368,234.661416,6.374831,7.056571,0.108425,-259.243116,0.112499,-69.119829,...,45.232593,146.1408,234.662576,6.375138,7.0566,-259.243,-69.120,46.840,2000.0,1.360083e-08
2,BD+083095,Gaia DR3 4454435622068380928,4454435622068380928,237.744492,8.422531,6.248225,0.014882,-236.858829,0.018193,-163.200020,...,43.674953,161.0899,237.745556,8.423256,6.2482,-236.859,-163.200,-51.217,2000.0,1.425101e-08
3,BD-002387,Gaia DR3 3803001194792721536,3803001194792721536,163.766577,-0.805174,22.230547,0.014925,-235.621190,0.016611,14.992534,...,50.463209,44.9224,163.767624,-0.805240,22.2305,-235.621,14.993,7.815,2000.0,7.100669e-09
4,BD-010184,Gaia DR3 2485808048791094784,2485808048791094784,21.288673,-1.060977,31.367302,0.018899,-193.863121,0.022558,-291.595323,...,-62.706157,31.6692,21.289535,-1.059681,31.3673,-193.863,-291.595,-5.212,2000.0,2.779448e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046,HIP7743,Gaia DR3 4954335455582406528,4954335455582406528,24.924153,-46.510346,28.467696,0.012047,17.928424,0.009886,-88.738965,...,-68.403263,NaN,24.924037,-46.509951,28.4677,17.928,-88.739,36.310,2000.0,1.389512e-08
1047,HIP78242,Gaia DR3 6040189774559716096,6040189774559716096,239.634577,-30.447761,17.197351,0.016537,-280.940148,0.018532,-219.683333,...,17.121064,57.8403,239.636025,-30.446784,17.1974,-280.940,-219.683,-64.087,2000.0,3.436649e-08
1048,HIP88316,Gaia DR3 6432643442021124864,6432643442021124864,270.544024,-69.342237,17.171234,0.012077,38.502806,0.007412,-326.956775,...,-21.081597,58.1295,270.543539,-69.340784,17.1712,38.503,-326.957,-81.008,2000.0,3.526269e-08
1049,HIP96240,Gaia DR3 6771373081565902208,6771373081565902208,293.541033,-23.016423,20.159183,0.014810,18.483638,0.015424,34.053341,...,-19.275750,49.5179,293.540944,-23.016574,20.1592,18.484,34.053,5.180,2000.0,2.337846e-09


In [6]:
harps_full['Star'] = harps_full['Star'].str.replace(' ', '')
harps_gaia_unique['target_id'] = harps_gaia_unique['target_id'].str.replace(' ', '')
harps_gaia_merged = pd.merge(harps_full, harps_gaia_unique, right_on='target_id', left_on='Star', how = 'inner')
harps_gaia_merged

,Star,Teff,e_Teff,logg,e_logg,fe_h,e_fe_h,vtur,e_vtur,target_id,...,b,distance_gspphot,target_ra,target_dec,target_parallax,target_pm_ra,target_pm_dec,target_radial_velocity,epoch,target_separation (deg)
0,HIP38324,4393,98.0,4.01,0.39,-0.32,0.03,0.44,0.29,HIP38324,...,-11.809382,32.4218,117.763938,-50.305003,30.8073,-140.793,163.940,-5.836,2000.0,1.102009e-08
1,HD85512,4400,45.0,4.36,0.10,-0.26,0.14,0.02,0.05,HD85512,...,8.159574,NaN,147.779383,-43.502784,88.6737,461.603,-471.880,-9.560,2000.0,1.085924e-07
2,HD192961,4418,51.0,4.23,0.18,-0.30,0.22,0.02,2.20,HD192961,...,-34.095905,18.4031,304.903901,-46.428115,54.0441,-375.461,-102.174,32.813,2000.0,7.294731e-08
3,HD218511,4424,90.0,4.52,0.26,-0.05,0.02,0.26,0.45,HD218511,...,-46.594034,NaN,347.420627,-67.732822,67.3957,-318.045,-214.068,4.560,2000.0,1.019950e-07
4,HIP25612,4433,95.1,4.33,0.35,-0.49,0.06,0.30,0.55,HIP25612,...,-29.766294,30.3643,82.076321,-81.775380,32.9372,-29.644,-441.140,50.037,2000.0,6.035343e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046,HD66740,6666,39.0,4.49,0.06,0.04,0.02,1.70,0.05,HD66740,...,2.304030,55.9425,121.005452,-26.973323,17.8619,-24.719,42.189,34.520,2000.0,4.890260e-09
1047,HD141128,6758,48.0,4.67,0.06,0.07,0.03,1.65,0.07,HD141128,...,33.768179,51.5420,236.908752,-9.480442,19.3562,-74.127,16.900,-36.649,2000.0,9.335363e-09
1048,CD-2310879,6788,43.0,4.67,0.04,-0.24,0.02,1.82,0.09,CD-2310879,...,38.370097,176.9176,195.864711,-24.420035,5.6284,9.543,-28.267,-9.894,2000.0,2.226132e-09
1049,HD20852,6813,70.0,4.76,0.06,-0.35,0.04,2.34,0.22,HD20852,...,-56.768879,51.5636,50.185770,-26.287805,19.3657,36.331,139.808,15.517,2000.0,4.103092e-09


In [7]:
harps_gaia_merged.drop([
    'target_ra', 'target_dec', 'target_parallax',
       'target_pm_ra', 'target_pm_dec', 'target_radial_velocity', 'epoch',
       'target_separation (deg)', 'target_id', 'designation',], axis =1, inplace = True)

#### UW nad UVW velocities

In [8]:
# Define a function to apply gal_population to each row
def calculate_UVW(row):
    UVW_values =  UVW(row['ra'], row['dec'], row['pmra'], row['pmdec'], row['radial_velocity'], row['parallax'])
    return pd.Series(UVW_values, index=['U_lsr', 'V_lsr', 'W_lsr'])

harps_gaia_merged[['U_lsr', 'V_lsr', 'W_lsr']] = harps_gaia_merged.apply(lambda row: calculate_UVW(row), axis=1)


Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad
Please check the value of vrad


In [9]:
# Some targets do not have RV or distances. Removed
harps_gaia_merged.dropna(inplace=True)

In [10]:
# Define a function to apply gal_population to each row
def calculate_probabilities(row):
    probs =  gal_population(row['U_lsr'], row['V_lsr'], row['W_lsr'], char_vel='Mean')
    return probs[0]

# Apply the function to each row using apply with axis=1
harps_gaia_merged['P_gal'] = harps_gaia_merged.apply(calculate_probabilities, axis=1)

In [11]:
harps_mg_si = pd.read_csv('harps_mg_si.tsv', sep = '\t')
harps_mg_si.drop(['al_Fe'], axis = 1, inplace = True)
harps_mg_si['Star'] = harps_mg_si['Star'].str.strip()

In [12]:
harps_gaia_mg_si = harps_gaia_merged.merge(harps_mg_si, on = 'Star')
harps_gaia_mg_si.head(3)

,Star,Teff,e_Teff,logg,e_logg,fe_h,e_fe_h,vtur,e_vtur,source_id,...,b,distance_gspphot,U_lsr,V_lsr,W_lsr,P_gal,mg_fe,e_mg_fe,si_fe,e_si_fe
0,HIP38324,4393,98.0,4.01,0.39,-0.32,0.03,0.44,0.29,5517008984911824896,...,-11.809382,32.4218,20.471708,22.675343,1.569178,98.9,-0.08,0.05,0.06,0.10
1,HD192961,4418,51.0,4.23,0.18,-0.30,0.22,0.02,2.20,6669657123437047424,...,-34.095905,18.4031,-54.443926,-4.640859,15.465497,97.8,-0.22,0.31,0.15,0.12
2,HIP25612,4433,95.1,4.33,0.35,-0.49,0.06,0.30,0.55,4621352894595893632,...,-29.766294,30.3643,-88.196035,-8.152206,-5.252849,95.4,0.06,0.11,0.22,0.14


In [13]:

def calculate_X_H(df, element):
    # Create column names dynamically
    X_FE_col = f'{element}_fe'
    X_FE_err_col = f'e_{element}_fe'
    FE_H_col = 'fe_h'
    FE_H_err_col = 'e_fe_h'
    
    # Compute X_H
    df[f'{element}_h'] = df[X_FE_col] + df[FE_H_col]
    
    # Propagate the uncertainties
    df[f'e_{element}_h'] = np.sqrt(df[X_FE_err_col]**2 + 0)
    
    return df

harps_gaia_mg_si = calculate_X_H(harps_gaia_mg_si, 'mg')
harps_gaia_mg_si = calculate_X_H(harps_gaia_mg_si, 'si')
harps_gaia_mg_si

,Star,Teff,e_Teff,logg,e_logg,fe_h,e_fe_h,vtur,e_vtur,source_id,...,W_lsr,P_gal,mg_fe,e_mg_fe,si_fe,e_si_fe,mg_h,e_mg_h,si_h,e_si_h
0,HIP38324,4393,98.0,4.01,0.39,-0.32,0.03,0.44,0.29,5517008984911824896,...,1.569178,98.9,-0.08,0.05,0.06,0.10,-0.40,0.05,-0.26,0.10
1,HD192961,4418,51.0,4.23,0.18,-0.30,0.22,0.02,2.20,6669657123437047424,...,15.465497,97.8,-0.22,0.31,0.15,0.12,-0.52,0.31,-0.15,0.12
2,HIP25612,4433,95.1,4.33,0.35,-0.49,0.06,0.30,0.55,4621352894595893632,...,-5.252849,95.4,0.06,0.11,0.22,0.14,-0.43,0.11,-0.27,0.14
3,HIP39470,4440,61.7,4.24,0.27,-0.32,0.06,0.14,0.28,5545635148103012096,...,1.180027,99.0,-0.30,0.21,0.06,0.14,-0.62,0.21,-0.26,0.14
4,HIP59925,4450,69.8,4.12,0.29,-0.25,0.01,0.15,0.53,3489077813176242432,...,-3.283166,98.9,-0.01,0.10,0.10,0.11,-0.26,0.10,-0.15,0.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
936,HD66740,6666,39.0,4.49,0.06,0.04,0.02,1.70,0.05,5697067105030910336,...,9.059767,98.9,-0.17,0.03,0.00,0.08,-0.13,0.03,0.04,0.08
937,HD141128,6758,48.0,4.67,0.06,0.07,0.03,1.65,0.07,4347491142557695360,...,0.565507,99.0,-0.12,0.04,-0.03,0.12,-0.05,0.04,0.04,0.12
938,CD-2310879,6788,43.0,4.67,0.04,-0.24,0.02,1.82,0.09,3497849373345037824,...,-17.473191,98.6,0.03,0.04,0.05,0.09,-0.21,0.04,-0.19,0.09
939,HD20852,6813,70.0,4.76,0.06,-0.35,0.04,2.34,0.22,5061776016700293376,...,2.122155,98.8,0.02,0.05,0.07,0.10,-0.33,0.05,-0.28,0.10


In [14]:
criteria_abundance = (
    (harps_gaia_mg_si['fe_h'] > -1) &
    (harps_gaia_mg_si['fe_h'] < 0.5) &
    (harps_gaia_mg_si['e_fe_h'] <= 0.05) &
    (harps_gaia_mg_si['mg_h'] > -1) &
    (harps_gaia_mg_si['mg_h'] < 0.5) & 
    (harps_gaia_mg_si['e_mg_h'] <= 0.1) & 
    (harps_gaia_mg_si['si_h'] > -1) &
    (harps_gaia_mg_si['si_h'] < 0.5) &
    (harps_gaia_mg_si['e_si_h'] <= 0.1)
    )

harps_gaia_mg_si = harps_gaia_mg_si[criteria_abundance]
len(harps_gaia_mg_si)

818

In [15]:
criteria_dwarfs = (harps_gaia_mg_si['Teff'] > 4500) & (harps_gaia_mg_si['Teff'] < 6500) & (harps_gaia_mg_si['logg'] > 4)
harps_gaia_mg_si = harps_gaia_mg_si[criteria_dwarfs]
len(harps_gaia_mg_si)

798

In [16]:
relevant_columns = [
    'Star', 'source_id',
    'ra', 'dec',
    'parallax', 'parallax_error',
    'pmra', 'pmra_error',
    'pmdec', 'pmdec_error',
    'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag',
    'radial_velocity', 'radial_velocity_error',
    'distance_gspphot',
    'U_lsr', 'V_lsr', 'W_lsr', 'P_gal',
    'Teff', 'e_Teff', 'logg', 'e_logg', 'vtur', 'e_vtur',   
    'fe_h', 'e_fe_h',
    'mg_h', 'e_mg_h', 
    'si_h', 'e_si_h', 
]
harps_gaia_mg_si = harps_gaia_mg_si[relevant_columns]

In [17]:
harps_gaia_mg_si

,Star,source_id,ra,dec,parallax,parallax_error,pmra,pmra_error,pmdec,pmdec_error,...,logg,e_logg,vtur,e_vtur,fe_h,e_fe_h,mg_h,e_mg_h,si_h,e_si_h
11,BD-012505,3790755044736796544,169.304406,-1.981918,30.138378,0.023513,-568.299830,0.017789,-14.309893,0.014746,...,4.13,0.38,0.24,0.72,-0.07,0.02,-0.13,0.10,0.09,0.09
21,HD55,4706630501049679744,1.322329,-67.835081,61.006400,0.019343,-116.617908,0.019904,-560.587881,0.018929,...,4.54,0.21,0.06,1.63,-0.67,0.01,-0.55,0.07,-0.46,0.08
25,HD42505,5480937375656689408,91.683263,-62.681825,35.384393,0.013205,63.838962,0.018247,176.640253,0.015244,...,4.24,0.27,0.25,0.50,-0.16,0.05,-0.21,0.10,-0.13,0.08
26,HD154387,5938742337829520256,256.880988,-48.248481,36.527404,0.018006,26.332181,0.021347,-35.901501,0.019008,...,4.15,0.50,0.28,0.41,-0.28,0.04,-0.47,0.07,-0.25,0.08
28,BD-130321,2460132463713337472,26.931812,-12.769909,27.501795,0.018170,253.272569,0.016348,167.272805,0.012531,...,4.20,0.32,0.39,0.40,0.08,0.04,-0.03,0.10,0.20,0.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,HD130989,6282232965755268992,222.895581,-17.790937,18.368855,0.033474,-14.522487,0.038095,-128.212972,0.042059,...,4.27,0.05,1.85,0.06,-0.23,0.03,-0.23,0.03,-0.15,0.10
924,HD149200,4354366423123745280,248.347131,-4.262206,19.431717,0.021556,-64.106993,0.025123,-50.786472,0.018299,...,4.64,0.05,1.74,0.06,0.15,0.03,0.03,0.03,0.16,0.07
925,HD135468,5903867203362296704,229.228173,-46.202477,18.076261,0.022241,-132.679688,0.023094,-26.904218,0.023107,...,4.25,0.05,1.82,0.05,-0.02,0.03,-0.02,0.03,0.00,0.08
926,HD112283,3679210174214660992,193.919957,-3.550592,18.549359,0.024745,-119.892920,0.031130,16.254477,0.029771,...,4.84,0.05,1.86,0.15,-0.13,0.05,-0.22,0.05,-0.18,0.07


In [18]:
harps_gaia_mg_si.to_csv('harps_gaia_mg_si.csv', index=False)

In [19]:
harps_c_o = pd.read_csv('harps_c_delgadomena2021.rdb', sep=r'\s+')

In [20]:
harps_c = harps_c_o[(harps_c_o['cfe_err'] < 0.1)]
harps_c['c_h'] = harps_c['cfe'] + harps_c['feh']
harps_c['e_c_h'] = harps_c['cfe_err']
harps_c = harps_c[['Star', 'c_h', 'e_c_h']]
harps_c['Star'] = harps_c['Star'].str.replace(' ', '')
len(harps_c)

C:\Users\adbva\AppData\Local\Temp\ipykernel_19360\1044510351.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  harps_c['c_h'] = harps_c['cfe'] + harps_c['feh']
C:\Users\adbva\AppData\Local\Temp\ipykernel_19360\1044510351.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  harps_c['e_c_h'] = harps_c['cfe_err']


743

In [21]:
harps_gaia_c = harps_c.merge(harps_gaia_mg_si, on = 'Star')
harps_gaia_c.to_csv('harps_gaia_c.csv', index=False)

In [22]:
harps_o = harps_c_o[harps_c_o['O_6158_err'] < 0.1]
harps_o['o_h'] = harps_o['O_6158'] + harps_o['feh']
harps_o['e_o_h'] = harps_o['O_6158_err']
harps_o = harps_o[['Star', 'o_h', 'e_o_h']]
harps_gaia_o = harps_o.merge(harps_gaia_mg_si, on = 'Star')
harps_gaia_o.to_csv('harps_gaia_o.csv', index=False)
len(harps_gaia_o)

C:\Users\adbva\AppData\Local\Temp\ipykernel_19360\3300472478.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  harps_o['o_h'] = harps_o['O_6158'] + harps_o['feh']
C:\Users\adbva\AppData\Local\Temp\ipykernel_19360\3300472478.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  harps_o['e_o_h'] = harps_o['O_6158_err']


232